Esse notebook se trata sobre um mini curso de aprendizado de máquinna que lecionei. Nele temos a implementação da regressão linear de forma iterativa e também a aplicação da regressão linear e logística da biblioteca sklearn. Há também algumas práticas como:

* Métricas do modelo.
* Desde a divisão do dataset.
* Separação de features e target.
* Validação do modelo.
* Regularização.
* Normalização.
* Aplicação com um exemplo novo.
* Evolução dos valores de θ que o modelo aprende.
* Overfitting e Underfitting.
* Predição vs Valores reais.
* Curva de erro.

Na regressão linear utilizei o dataset do preço das casas de acordo com as caracteristicas da casa, já na regerssão logística utilizei o dataset do titanic que prevê se um passageiro irá sobreviver ou não. Utilizei eemplos simples pois se tratava de um curso de introdução e pela limitação do tempo também. Ambos retirados do site kaggle.

Cada célula possuí um título com a sua responsabilidade.

# Regressão linear iterativa

# 01) Célula Download do dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("altavish/boston-housing-dataset")

print("Path to dataset files:", path)

# 02) Célula Importação das bibliotecas auxiliares

Gostaria de fazer um adendo à célula abaixo referente a importação das bibliotecas. Abaixo faço a importação das bibliotecas que serão usadas em todo o notebook. Mas em alguns outros lugares faço a importaçao de outras partes mais específicas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# 03) Célula Importação do dataset e verificação se há valores faltantes

In [ ]:
#Importação do CSV como um dataset
df = pd.read_csv("/root/.cache/kagglehub/datasets/altavish/boston-housing-dataset/versions/1/HousingData.csv")

print("Dimenções:", df.shape)
print("Valores faltantes antes do tratamento:")
print(df.isnull().sum())

# Tratamento de valores faltantes e reenchendo os valores nulos com média

df.fillna(df.mean(), inplace=True)

print("\nValores faltantes após o tratamento:")
print(df.isnull().sum())

df.head()

# 04) Célula Classe do gradiente

In [ ]:
class RTrainer:
    def __init__(self, alfa=0.01, iteracoes=50000, lambda_ridge=np.log(100)):
        self.alfa = alfa
        self.iteracoes = iteracoes
        self.lambda_ridge = lambda_ridge  # lambda da regularização
        self.theta = None
        self.training_loss = []
        self.validation_loss = []
        self.train_time = 0

    def fit(self, X, y, X_val=None, y_val=None):
        start = time.time()

        # Adiciona coluna de 1s para bias
        X = np.c_[np.ones(X.shape[0]), X]
        n_amostras, n_atributos = X.shape
        self.theta = np.zeros(n_atributos)

        self.theta_valores = []

        for _ in range(self.iteracoes):
            y_pred = X.dot(self.theta)
            error = y_pred - y

            # Gradiente com Ridge (não regularizamos o bias θ0)
            grad = (1/n_amostras) * (X.T.dot(error) + self.lambda_ridge * np.r_[0, self.theta[1:]])

            # Atualização
            self.theta -= self.alfa * grad

            # Loss treino (MSE + regularização Ridge)
            loss_train = np.mean(error**2) + (self.lambda_ridge/(2*n_amostras)) * np.sum(self.theta[1:]**2)
            self.training_loss.append(loss_train)

            # Loss validação (se fornecido)
            if X_val is not None and y_val is not None:
                X_val_bias = np.c_[np.ones(X_val.shape[0]), X_val]
                y_val_pred = X_val_bias.dot(self.theta)
                error_val = y_val_pred - y_val
                loss_val = np.mean(error_val**2) + (self.lambda_ridge/(2*len(y_val))) * np.sum(self.theta[1:]**2)
                self.validation_loss.append(loss_val)

            self.theta_valores.append(self.theta.copy())

        end = time.time()
        self.train_time = end - start
        print(f"Tempo que rodou: {self.train_time:.2f} segundos.")
        return self.theta

    def predict(self, X):
        X = np.c_[np.ones(X.shape[0]), X]
        return X.dot(self.theta)



# 05) Célula Classe referente as métricas do modelo

In [ ]:
class Metricas:
    # Erro médio absoluto
    def mae(y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred))

    # Erro quadrático absoluto
    def mse(y_true, y_pred):
        return np.mean((y_true - y_pred)**2)

    # Erro médio de porcentagem absoluto
    def mape(y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


# 06) Célula Separação dos dados para o target

In [ ]:
#Pegando os valores do dataset
#X features, Y target
#.values para conversão para numpy

X = df.drop('MEDV', axis=1).values
y = df['MEDV'].values

print("Dimensão de X, features:", X.shape)
print("Dimensão de y, target:", y.shape)

# 07) Validação

In [ ]:
#Dividindo o dataset em treino,validação e teste

# Define seed para reprodutibilidade
np.random.seed(42)

# Embaralha os índices para tentar evitar que fique enviesado
indices = np.arange(len(X))
np.random.shuffle(indices)

# Calcula cortes
n_total = len(X)
n_train = int(0.6 * n_total)
n_dev   = int(0.2 * n_total)

# Divide manualmente

#Treino
X_train, y_train = X[indices[:n_train]], y[indices[:n_train]]
#Validação
X_dev, y_dev     = X[indices[n_train:n_train+n_dev]], y[indices[n_train:n_train+n_dev]]
#Teste em si
X_test, y_test   = X[indices[n_train+n_dev:]], y[indices[n_train+n_dev:]]

print("Treino:", X_train.shape)
print("Desenvolvimento:", X_dev.shape)
print("Teste:", X_test.shape)

# 08) Célula Normalização do dados para evitar erros e incoerências

In [ ]:

# Normalização manual (sem sklearn)
X_mean = np.mean(X_train, axis=0)
X_std = np.std(X_train, axis=0)

# Evitar divisão por zero se alguma feature tiver desvio padrão zero
# (Embora improvável com dados reais, é uma boa prática)
# X_std[X_std == 0] = 1e-8

X_train = (X_train - X_mean) / X_std
X_dev   = (X_dev   - X_mean) / X_std
X_test  = (X_test  - X_mean) / X_std


# 09) Célula Ajustes de parâmetros do modelo/mudei aqui

In [ ]:
valores_alfas = [0.00001, 0.0001, 0.001, 0.01] # Ajustando as taxas de aprendizado
melhor_alfa = None
melhorErroModelo = float("inf")

for alfas in valores_alfas:
    # Alteração do número de iteração mexxem com convergência com taxas menores
    # Como o número de iterações muda a precisão do modelo
    # Não esquecer de testar os falores de iterações
    modelo = RTrainer(alfa=alfas, iteracoes=20000)
    # Chamando a função de treino definido lá em cima
    modelo.fit(X_train, y_train)
    # Treinando o modelo com a função predict definida lá em cima também
    y_pred_dev = modelo.predict(X_dev)
    erroModelo = Metricas.mse(y_dev, y_pred_dev)
    print(f"Alfa={alfas} -> Dev MSE={erroModelo:.4f}")

    if erroModelo < melhorErroModelo and not np.isnan(erroModelo): # Adiciona verificação para NaN
        melhorErroModelo = erroModelo
        melhor_alfa = alfas

print("\nMelhor taxa de aprendizado encontrada:", melhor_alfa)

# 10) Célula Parte referente a regularização, usando Ridge

In [ ]:

lambdas = np.concatenate([[0.0], np.logspace(-6, 3, 10)])
dev_mses = []
coef_norms = []

for lam in lambdas:
    # cria e treina o modelo (passe X_dev/y_dev para validação dentro do fit)
    model = RTrainer(alfa=melhor_alfa if 'melhor_alfa' in globals() else 0.01,
                     iteracoes=2000,
                     lambda_ridge=lam)
    # fornece X_dev,y_dev na fit para calcular validation_loss (se seu fit usa esses argumentos)
    model.fit(X_train, y_train, X_val=X_dev, y_val=y_dev)

    # calcular MSE no dev a partir do vetor guardado de validation_loss (último valor)
    if hasattr(model, 'validation_loss') and model.validation_loss:
        dev_mse = model.validation_loss[-1]
    else:
        # fallback: calcular diretamente
        y_dev_pred = model.predict(X_dev)
        dev_mse = np.mean((y_dev - y_dev_pred)**2)
    dev_mses.append(dev_mse)

    # norma dos coeficientes (ignorando bias theta[0])
    theta = model.theta
    coef_norms.append(np.linalg.norm(theta[1:], ord=2))

    print(f"lambda={lam:.6g} -> dev MSE={dev_mse:.6f}, ||theta||_2={coef_norms[-1]:.6f}")

# 11) Célula Tempo que o algortimo rodou com os dados para serem previsto.

In [ ]:
if melhor_alfa is not None:
    X_full = np.vstack([X_train, X_dev])
    y_full = np.concatenate([y_train, y_dev])

    final_model = RTrainer(alfa=melhor_alfa, iteracoes=50000)
    final_model.fit(X_full, y_full)
    #final_model.plot_losses()

    y_pred_test = final_model.predict(X_test)

    #Print dos valores de theta
    theta_final = final_model.theta
    print("Valores finais de θ:", theta_final)

    #Criar equação da regressão
    features = df.drop("MEDV", axis=1).columns  #nomes das features
    equacao = f"y = {theta_final[0]:.4f}"       #intercepto

    for coef, nome in zip(theta_final[1:], features):
        sinal = " + " if coef >= 0 else " - "
        equacao += f"{sinal}{abs(coef):.4f}*{nome}"

    print("\nEquação da regressão linear:")
    print(equacao)

    print("Métricas no Teste")
    print(f"MAE: {Metricas.mae(y_test, y_pred_test):.10}")
    print(f"MSE: {Metricas.mse(y_test, y_pred_test):.10}")
    print(f"MAPE: {Metricas.mape(y_test, y_pred_test):.10}")
    print(f"⏱ Tempo de treino: {final_model.train_time:.2} segundos")
else:
    print("Não foi possível encontrar uma melhor taxa de aprendizado. O modelo final não foi treinado.")

#12) Célula Verificando se há ou não underfitting pelo MSE

In [ ]:
# Avaliação no conjunto de Treinamento
y_pred_train = final_model.predict(X_full) # Previsões no conjunto de treino + desenvolvimento

print("Métricas no Conjunto de Treinamento:")
print(f"MAE: {Metricas.mae(y_full, y_pred_train):.6}")
print(f"MSE: {Metricas.mse(y_full, y_pred_train):.6}")
print(f"MAPE: {Metricas.mape(y_full, y_pred_train):.6}")

print("\nMétricas no Conjunto de Teste:")
# As métricas no conjunto de teste já foram calculadas e impressas no bloco anterior (Qcin9TL3_6sj)
# Vamos apenas exibir as variáveis y_test e y_pred_test para referência
print(f"MAE: {Metricas.mae(y_test, y_pred_test):.6}")
print(f"MSE: {Metricas.mse(y_test, y_pred_test):.6}")
print(f"MAPE: {Metricas.mape(y_test, y_pred_test):.6}")

# Análise de Overfitting vs Underfitting (Exemplo de como interpretar)
# Compare as métricas de Treinamento e Teste:
# - Se as métricas de Treinamento forem significativamente melhores que as de Teste -> Overfitting
# - Se as métricas de Treinamento e Teste forem ambas ruins -> Underfitting
# - Se as métricas de Treinamento e Teste forem próximas e boas -> Bom ajuste

# Exemplo de Análise Textual (você deve escrever sua própria análise baseada nos resultados)
if Metricas.mse(y_full, y_pred_train) < Metricas.mse(y_test, y_pred_test):
    print("\nAnálise: O MSE no conjunto de treinamento é menor que no conjunto de teste. Isso pode indicar overfitting.")
else:
    print("\nAnálise: O MSE no conjunto de treinamento é maior ou igual ao do conjunto de teste. O modelo pode estar com underfitting ou bem ajustado.")

#13) Célula Predição vs Valores Reais (Conjunto de Treino + Desenvolvimento)

In [ ]:
# Certifique-se de que y_full e y_pred_train foram calculados (executando o bloco anterior)
# y_full = np.concatenate([y_train, y_dev])
# y_pred_train = final_model.predict(X_full)

plt.figure(figsize=(8, 6))
plt.scatter(y_full, y_pred_train, color="blue", alpha=0.5) # Use alpha para ver a densidade dos pontos
plt.plot([y_full.min(), y_full.max()], [y_full.min(), y_full.max()], 'r--')  # linha y=x
plt.xlabel("Valores Reais (Treino + Desenvolvimento)")
plt.ylabel("Predição (Treino + Desenvolvimento)")
plt.title("Predição vs Valores Reais (Conjunto de Treino + Desenvolvimento)")
plt.grid(True)
plt.show()

#14) Célula Curva de erro com o passar das iterações

In [ ]:
if final_model is not None and hasattr(final_model, 'training_loss') and final_model.training_loss:
    plt.plot(final_model.training_loss)
    plt.xlabel("Iterações")
    plt.ylabel("Erro (MSE)")
    plt.title("Curva de Treinamento")
    plt.grid(True)
    plt.show()
    print("\nPrimeiros 10 valores da perda de treinamento:", final_model.training_loss[:10])
    print("Últimos 10 valores da perda de treinamento:", final_model.training_loss[-10:])
else:
    print("Não há dados de perda de treinamento para plotar.")

#Dar uma olhada aqui:https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent?hl=pt-br

#15) Célula Gráfico dos valores de θ

In [ ]:
# Converte para numpy array para facilitar o plot
theta_hist = np.array(final_model.theta_valores)

# Cada linha = iteração, cada coluna = um θ
plt.figure(figsize=(12,6))
for i in range(theta_hist.shape[1]):
    plt.plot(theta_hist[:, i], label=f"θ{i}")
plt.xlabel("Iterações")
plt.ylabel("Valor de θ")
plt.title("Evolução dos Parâmetros θ durante o Treinamento")
plt.grid(True)
plt.legend()
plt.show()


#16) Célula Outro gráfico dos valores de θ

In [ ]:
#Outro gráfico para medir o valor de theta
coef = final_model.theta[1:]
plt.bar(range(len(coef)), coef)
plt.xticks(range(len(coef)), df.drop("MEDV", axis=1).columns, rotation=90)
plt.title("Coeficientes do Modelo")
plt.grid(True)
plt.show()

#17) Célula Gráficos dos valores reais e predição

In [ ]:
plt.scatter(y_test, y_pred_test, color="purple")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # linha y=x
plt.xlabel("Valores Reais")
plt.ylabel("Predição")
plt.title("Predição vs Valores Reais")
plt.grid(True)
plt.show()


# 18) célula Um exemplo "novo" fora dos dados

In [ ]:
# Exemplo de novo X (valores crus)
novo_X = np.array([[0.1, 18.0, 2.0, 0, 0.5, 6.0, 45.0, 2.0, 3, 300, 15, 390, 5]])

# Normaliza usando média e desvio padrão do treino
novo_X_norm = (novo_X - X_mean) / X_std

# Faz a predição
y_pred_novo = final_model.predict(novo_X_norm)
print("Preço estimado da casa para o novo exemplo:", y_pred_novo[0])


# Regressão linear usando o scikit learn

## Importação e configuração

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

# --- CARREGAMENTO DO DATASET ---
df = pd.read_csv('/kaggle/input/boston-housing-dataset/HousingData.csv')
print(f"Dataset Carregado Originalmente: {df.shape[0]} linhas e {df.shape[1]} colunas")

In [ ]:
df.head()

In [ ]:
df.info()

## Preparação e limpeza dos dados

In [ ]:
if not df.empty:
    nulos = df.isnull().sum().sum()
    if nulos > 0:
        print(f"\nAVISO: Encontrados {nulos} valores nulos no dataset.")
        print("Preenchendo valores faltantes com a média das colunas...")
        df.fillna(df.mean(numeric_only=True), inplace=True)
        print(f"Shape após preenchimento: {df.shape}")

    # Definição de X (Features) e y (Target)
    # MEDV = Median Value of Owner-Occupied homes in $1000's (Nosso alvo)
    target_col = 'MEDV'

    # Se o dataset usar nomes diferentes, ajustamos aqui. O padrão é MEDV.
    if target_col not in df.columns:
        # Tenta achar a coluna alvo pelo índice se o nome mudar (geralmente é a última)
        target_col = df.columns[-1]

    X = df.drop(target_col, axis=1)
    y = df[target_col]

    feature_names = X.columns.tolist()

    # Divisão Treino (80%) e Teste (20%) hold-out
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # NORMALIZAÇÃO
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\nDados separados, limpos e normalizados!")

In [ ]:
df.info()

## Treinamento, regularização e validação

In [ ]:
if not df.empty:
    modelos = {
        "Linear Simples": LinearRegression(),
        "Ridge (L2)": Ridge(alpha=1.0), # Penaliza coeficientes altos
        "Lasso (L1)": Lasso(alpha=0.1)  # Tenta zerar coeficientes irrelevantes
    }

    results = {}

    print("\n" + "="*50)
    print("AVALIAÇÃO DOS MODELOS (Previsão de Preço de Casas x1000$)")
    print("="*50)

    for nome, model in modelos.items():
        # Treino
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        # Métricas
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        # Validação Cruzada
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')

        results[nome] = {'R2': r2, 'RMSE': rmse, 'Model': model}

        print(f"\n--- {nome} ---")
        print(f"R² (Teste): {r2:.4f}")
        print(f"R² Médio (CV): {cv_scores.mean():.4f}")
        print(f"Erro Médio (MAE): ${mae*1000:.2f}") # Multiplicando por 1000 pois MEDV é em x1000
        print(f"RMSE: {rmse:.4f}")

## A equação final (interpretação dos pesos)

In [ ]:
    best_model = results['Linear Simples']['Model']
    intercepto = best_model.intercept_
    coeficientes = best_model.coef_

    print("\n" + "="*50)
    print("O QUE VALORIZA UMA CASA EM BOSTON?")
    print("="*50)
    print(f"Preço Base (Intercepto): {intercepto:.2f} (mil dólares)")

    coef_df = pd.DataFrame({'Feature': feature_names, 'Impacto (Peso)': coeficientes})

    coef_df['Importância Absoluta'] = coef_df['Impacto (Peso)'].abs()
    coef_df = coef_df.sort_values(by='Importância Absoluta', ascending=False)

    display(coef_df[['Feature', 'Impacto (Peso)']])

    print("\nANÁLISE:")
    print("- RM (Número de quartos) deve ter impacto positivo alto.")
    print("- LSTAT (População de baixa renda) geralmente tem impacto negativo forte.")
    print("- DIS (Distância do centro) e NOX (Poluição) também são relevantes.")

## Visualização

In [ ]:
    y_pred_final = best_model.predict(X_test_scaled)

    # Gráfico 1: Real vs Predito
    plt.figure(figsize=(7, 6)) # Mantendo a proporção, mas separando a figura
    sns.scatterplot(x=y_test, y=y_pred_final, alpha=0.7, edgecolor='k')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Preço Real (MEDV)')
    plt.ylabel('Preço Previsto')
    plt.title('Precisão da Regressão')
    plt.grid(True) # Adicionando grid para melhor visualização
    plt.show() # Mostrar a primeira figura

# Teste novo (fora do conjunto)

In [ ]:
    print("\n" + "="*50)
    print("SIMULADOR DE PREÇO DE IMÓVEL")
    print("="*50)

    # Dicionário com médias
    nova_casa = {
        'CRIM': 0.1,    # Taxa de criminalidade baixa
        'ZN': 20.0,     # Terreno residencial
        'INDUS': 5.0,   # Pouca indústria perto
        'CHAS': 0,      # Longe do rio (0 ou 1)
        'NOX': 0.5,     # Poluição média
        'RM': 7.0,      # 7 quartos (Casa grande!)
        'AGE': 50.0,    # 50 anos de construção
        'DIS': 4.0,     # Distância média dos centros de emprego
        'RAD': 5,       # Acesso a rodovias
        'TAX': 300,     # Imposto
        'PTRATIO': 15,  # Relação aluno/professor (escolas boas tem número baixo)
        'B': 390.0,     # Índice demográfico (Black proportion - variável polêmica histórica desse dataset)
        'LSTAT': 5.0    # Baixa porcentagem de população de baixa renda (bairro rico)
    }

    df_casa = pd.DataFrame([nova_casa])

    # Garantir ordem
    df_casa = df_casa[feature_names]

    # Normalizar
    df_casa_scaled = scaler.transform(df_casa)

    # Prever
    preco_previsto = best_model.predict(df_casa_scaled)

    print("CARACTERÍSTICAS DA CASA:")
    print(f"Quartos: {nova_casa['RM']} | Criminalidade: {nova_casa['CRIM']} | Bairro Rico (LSTAT baixo): {nova_casa['LSTAT']}%")
    print(f"PREVISÃO DE PREÇO: ${preco_previsto[0]*1000:.2f}")

# Regresão logística

## Importação e configuração

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/titanic-dataset")

print("Path to dataset files:", path)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Configuração visual
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

# --- CARREGAMENTO ---
try:
    df = pd.read_csv('/kaggle/input/titanic-dataset/Titanic-Dataset.csv')
    print(f"Dataset Titanic Carregado: {df.shape[0]} passageiros.")
except FileNotFoundError:
    print("ERRO: Arquivo 'Titanic-Dataset.csv' não encontrado.")
    df = pd.DataFrame()

In [ ]:
df.head()

# Limpeza e preparação dos dados

In [ ]:
if not df.empty:
    # 1. Remover colunas que não ajudam na matemática simples (Nomes, Tickets, Cabine cheia de nulos)
    cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df_clean = df.drop(cols_to_drop, axis=1)

    # 2. Tratar Valores Nulos (Missing Data)
    # Idade: Vamos preencher os vazios com a média
    df_clean['Age'].fillna(df_clean['Age'].mean(), inplace=True)
    # Embarked: Vamos preencher os 2 vazios com a moda (o porto mais comum)
    df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)

    # 3. Encoding (Texto -> Números)
    # Sex: male/female -> 0/1. Drop_first=True cria uma coluna só (ex: Sex_male: 1 ou 0)
    # Embarked: C/Q/S -> Cria colunas separadas
    df_processed = pd.get_dummies(df_clean, columns=['Sex', 'Embarked'], drop_first=True)

    # Definição de X e y
    X = df_processed.drop('Survived', axis=1)
    y = df_processed['Survived']

    feature_names = X.columns.tolist()

    # Divisão Treino/Teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # NORMALIZAÇÃO
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("Dados limpos: Normalização, conversão de valores etc")

## Treino do modelo

In [ ]:
# ==============================================================================
# 3. TREINAMENTO E REGULARIZAÇÃO
# ==============================================================================
if not df.empty:

    model = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    # Previsões
    y_pred = model.predict(X_test_scaled)
    # Probabilidades (Para ver a certeza do modelo: 0.80, 0.95, etc)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

## Avalição (as métricas da classificação)

In [ ]:
    acuracia = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')

    print("\n" + "="*50)
    print(f"RESULTADOS DO MODELO")
    print("="*50)
    print(f"Acurácia no Teste: {acuracia*100:.2f}% (Acertou {acuracia*100:.0f}% dos passageiros)")
    print(f"Acurácia Média (Validação Cruzada): {cv_scores.mean()*100:.2f}%")

    print("\nRELATÓRIO DETALHADO:")
    print(classification_report(y_test, y_pred))

    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Morreu (Pred)', 'Viveu (Pred)'],
                yticklabels=['Morreu (Real)', 'Viveu (Real)'])
    plt.title('Matriz de Confusão')
    plt.show()

# Os valores de θ (coeficientes)

In [ ]:
    coefs = model.coef_[0]

    print("\n" + "="*50)
    print("O QUE DETERMINA A SOBREVIVÊNCIA?")
    print("="*50)

    coef_df = pd.DataFrame({'Fator': feature_names, 'Peso (Log-Odds)': coefs})
    # Ordenar por importância (valor absoluto)
    coef_df['Impacto Absoluto'] = coef_df['Peso (Log-Odds)'].abs()
    coef_df = coef_df.sort_values(by='Impacto Absoluto', ascending=False)

    display(coef_df[['Fator', 'Peso (Log-Odds)']])

    print("\nINTERPRETAÇÃO:")
    print("- Peso POSITIVO: Aumenta a chance de sobreviver.")
    print("- Peso NEGATIVO: Diminui a chance (Aumenta risco de morrer).")
    print("Observe 'Sex_male'. Se for negativo forte, indica que homens tiveram menos chance.")

# Simulação: Jack vs Rose

In [ ]:
    print("\n" + "="*50)
    print("TESTE: JACK vs ROSE")
    print("="*50)

    # ROSE: 1ª Classe, 17 anos, Mulher (Sex_male=0), Pagou caro, Embarcou em Southampton
    rose = pd.DataFrame([{
        'Pclass': 1, 'Age': 17, 'SibSp': 0, 'Parch': 1, 'Fare': 100,
        'Sex_male': 0, 'Embarked_Q': 0, 'Embarked_S': 1
    }])

    # JACK: 3ª Classe, 20 anos, Homem (Sex_male=1), Pagou barato, Embarcou em Southampton
    jack = pd.DataFrame([{
        'Pclass': 3, 'Age': 20, 'SibSp': 0, 'Parch': 0, 'Fare': 7,
        'Sex_male': 1, 'Embarked_Q': 0, 'Embarked_S': 1
    }])

    # Normalizar os dados dos personagens usando o MESMO scaler do treino
    rose_scaled = scaler.transform(rose[feature_names])
    jack_scaled = scaler.transform(jack[feature_names])

    # Prever Probabilidades
    prob_rose = model.predict_proba(rose_scaled)[0][1] # Pega a prob da classe 1 (Sobreviver)
    prob_jack = model.predict_proba(jack_scaled)[0][1]

    print(f"Chance da ROSE sobreviver: {prob_rose*100:.2f}%")
    print(f"Chance do JACK sobreviver: {prob_jack*100:.2f}%")

    if prob_jack < 0.5:
        print("Resultado: Infelizmente, o modelo prevê que Jack não sobrevive (Limiar < 50%).")

# Visualização da curva sigmoide

In [ ]:
from scipy.special import expit # Função matemática da sigmoide

print("\n" + "="*50)
print("PLOTANDO A CURVA SIGMOIDE DO TITANIC")
print("="*50)

plt.figure(figsize=(10, 6))

# 1. Calcular o Score Linear
z_scores = model.decision_function(X_test_scaled)

# 2. Calcular as probabilidades baseadas nesse score (A curva S)

probs_sigmoide = expit(z_scores)

# 3. Plotar os passageiros REAIS
# Eixo X: O Score calculado pelo modelo

colors = ['red' if y == 0 else 'blue' for y in y_test]
plt.scatter(z_scores, y_test, c=colors, alpha=0.5, s=50, label='Dados Reais (Passageiros)', edgecolors='k')

# 4. Plot da LINHA DA SIGMOIDE

sorted_indices = np.argsort(z_scores)
z_sorted = z_scores[sorted_indices]
probs_sorted = probs_sigmoide[sorted_indices]

plt.plot(z_sorted, probs_sorted, color='orange', linewidth=3, label='Curva Sigmoide (Modelo)')

# 5.Limiar de Decisão (50%)
plt.axhline(y=0.5, color='green', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Fronteira de Decisão')


plt.title('Regressão Logística: Probabilidade de Sobrevivência', fontsize=15)
plt.xlabel('Score Linear (Combinação de Idade, Classe, Sexo, etc...)', fontsize=12)
plt.ylabel('Probabilidade P(Sobreviver)', fontsize=12)
plt.legend(loc='center right')
plt.grid(True, alpha=0.3)

# Mostrar onde estariam Jack e Rose nesse gráfico
# Recalcular score deles
z_rose = model.decision_function(rose_scaled)[0]
z_jack = model.decision_function(jack_scaled)[0]

plt.annotate('Rose', xy=(z_rose, expit(z_rose)), xytext=(z_rose-2, 0.9),
             arrowprops=dict(facecolor='blue', shrink=0.05))
plt.annotate('Jack', xy=(z_jack, expit(z_jack)), xytext=(z_jack+1, 0.2),
             arrowprops=dict(facecolor='red', shrink=0.05))

plt.show()